In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

%matplotlib inline

In [ ]:
# Task 1: Write your code here:


csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)
# df = df.drop(columns="Unnamed: 0", axis=1)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop("Order_ID",axis=1)


In [ ]:
# Task 2: Write your code here:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
#drop the rows with missing values in all columns since percanteges are low.
df = df.dropna(subset=['Delivery_Time', 'Weather', 'Traffic_Level',"Time_of_Day","Courier_Experience_yrs"])
df.info()
#lost about 300 rows! maybe filling some values was better



In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
df.info()

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
# df_forOneHot = df[categorical_cols]
# onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# df_encoded = pd.DataFrame(onehot_encoder.fit_transform(df_forOneHot), columns=onehot_encoder.get_feature_names_out(df_forOneHot.columns))
#dont know how to merge so..

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le



In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
check_target_distribution(df, "Delivery_Time")
# no target imbalance, target seems normally distributed

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
# since the data is not very imbalanced we can just do kfold with shuffling
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mae, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_losses = []
lr_mae = []

model = RandomForestRegressor(n_estimators=200)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # Calculate evaluation metrics
  mae = sklearn_mae(y_test, y_pred)


  # Store results
  lr_mae.append(mae)

print(np.mean(lr_mae))


In [ ]:
# Task 1: Write your code here:
importances = {}

importances['Random Forest'] = model.feature_importances_
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()




In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.scatter(y_test, y_pred, alpha=0.5, s=10, c='steelblue')

# Add perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('Actual Delivery(minutes) ', fontsize=12)
plt.ylabel('Predicted Delivery(minutes) ', fontsize=12)
plt.title('Predicted vs Actual ', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Task Bonus: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mae, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_losses = []
lr_mae = []

model1 = RandomForestRegressor(n_estimators=200)
model2 = CatBoostRegressor(verbose=0)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model1.fit(X_train, y_train) # train
  model2.fit(X_train,y_train)
  y_pred1 = model1.predict(X_test) # validate
  y_pred2 = model2.predict(X_test) # validate
  y_pred_avg = (y_pred1 + y_pred2) / 2


  # Calculate evaluation metrics
  mae = sklearn_mae(y_test, y_pred_avg)


  # Store results
  lr_mae.append(mae)

print(np.mean(lr_mae))